In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.4/398.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 94.7 MB/s eta 0:00:00
  Attempting uninstall: cython
    Found existing installation: Cython 3.0.12
    Uninstalling Cython-3.0.12:
      Successfully uninstalled Cython-3.0.12
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
xarray 2025.3.1 requires pandas

In [6]:
!pip install --upgrade --force-reinstall numpy pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 38.0 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2
    Uninstalling tzdata-2025.2:
      Successfully uninstalled tzdata-2025.2
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-

In [4]:
!pip install --upgrade dice-ml

In [22]:
# Ensure dependencies are installed BEFORE importing
# !pip install --upgrade --force-reinstall numpy pandas
# !pip install -r requirements.txt
# !pip install dice-ml

from IPython import get_ipython
from IPython.display import display
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import dice_ml
from dice_ml import Dice
import joblib
import lightgbm as lgb
import warnings
from lightgbm import LGBMRegressor

# Suppress warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

# Load dataset
data = pd.read_csv('/content/drive/MyDrive/DS Project/Hospital_Inpatient_Discharges__SPARCS_De-Identified___2022_20250423.csv')

# Data Cleanup
data['stay_120+'] = np.where(data['Length of Stay'] == '120 +', 1, 0)
data['Length of Stay'] = data['Length of Stay'].replace('120 +', '120').astype(int)
data['Total Charges'] = data['Total Charges'].str.replace(',', '').astype(float)
data['Total Costs'] = data['Total Costs'].str.replace(',', '').astype(float)

# Fill missing values
numeric_cols = data.select_dtypes(include=np.number).columns
for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

# Drop rows with any missing values
data = data.dropna(axis=0)

# Clean column names
def clean_col_name(col_name):
    col = col_name
    if "[" in col or "]" in col:
        col = col.replace('[', '').replace(']', '')
    if "<" in col:
        col = col.replace('<', 'less than')
    return col

# Clean all column names
data.columns = [clean_col_name(col) for col in data.columns]

# Define categorical columns (using cleaned names)
categorical_columns = [
    'Hospital Service Area', 'Hospital County', 'Facility Name', 'Age Group',
    'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Type of Admission',
    'Patient Disposition', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description',
    'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Description',
    'APR MDC Description', 'APR Severity of Illness Description',
    'APR Risk of Mortality', 'APR Medical Surgical Description',
    'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3',
    'Birth Weight', 'Emergency Department Indicator'
]

# Prepare features and target - EXCLUDE 'Total Charges'
X = data.drop(columns=['Total Costs', 'Total Charges'])
y = data['Total Costs']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Get actual categorical columns present in data
actual_categorical_columns = [col for col in categorical_columns if col in X_train.columns]

# Create mapping from category to integer code for each categorical feature
category_mappings = {}
for col in actual_categorical_columns:
    # Convert to string to handle mixed types
    X_train[col] = X_train[col].astype(str)
    categories = X_train[col].unique().tolist()
    category_mappings[col] = {category: idx for idx, category in enumerate(categories)}
    X_train[col] = X_train[col].map(category_mappings[col])

# Apply same mapping to test data
for col in actual_categorical_columns:
    X_test[col] = X_test[col].astype(str).map(category_mappings[col]).fillna(0)

# Prepare full DataFrame for DiCE (using training data only)
df_dice = pd.concat([X_train, y_train], axis=1)

# Create categorical dictionary with training set categories for DiCE
categorical_dict = {}
for col in actual_categorical_columns:
    # Get the original string categories
    original_categories = list(category_mappings[col].keys())
    categorical_dict[col] = original_categories

# Setup data for DiCE
data_reg = dice_ml.Data(
    dataframe=df_dice,
    continuous_features=[col for col in df_dice.columns
                         if col not in actual_categorical_columns and col != 'Total Costs'],
    categorical_features=categorical_dict,
    outcome_name='Total Costs'
)

# Train a new model with integer-coded categorical features
model = LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save the model for future use
model_path = '/content/drive/MyDrive/DS Project/lgb_model_integer_coded.pkl'
joblib.dump(model, model_path)

# Create sklearn wrapper for the LightGBM model
class LightGBMWrapper:
    def __init__(self, model, category_mappings):
        self.model = model
        self.category_mappings = category_mappings

    def predict(self, X):
        # Convert categorical columns to numeric codes
        X = X.copy()
        for col, mapping in self.category_mappings.items():
            if col in X.columns:
                # Convert to string and map to integers
                X[col] = X[col].astype(str).map(mapping)
                # Fill missing values with first category code
                X[col] = X[col].fillna(0).astype(int)
        return self.model.predict(X)

# Wrap the LightGBM model
wrapped_model = LightGBMWrapper(model, category_mappings)

# Create model interface for DiCE with sklearn backend
model_dice_reg = dice_ml.Model(model=wrapped_model, backend="sklearn", model_type='regressor')

# Initialize explainer
exp_reg = Dice(data_reg, model_dice_reg)

# Select query instance
query_instance_reg = X_test.iloc[[0]].copy()

# For DiCE, we need to convert back to original string representation
query_instance_dice = query_instance_reg.copy()
for col in actual_categorical_columns:
    if col in query_instance_dice.columns:
        # Reverse mapping from code to category
        reverse_mapping = {v: k for k, v in category_mappings[col].items()}
        value = query_instance_dice[col].iloc[0]
        if value in reverse_mapping:
            query_instance_dice[col] = reverse_mapping[value]
        else:
            query_instance_dice[col] = categorical_dict[col][0]

        # Ensure the value is in the training set categories
        if query_instance_dice[col].iloc[0] not in categorical_dict[col]:
            query_instance_dice[col] = categorical_dict[col][0]

# Features to vary
safe_features_to_vary = [col for col in X.columns if col not in [
    'CCSR Procedure Code', 'CCSR Procedure Description',
    'Facility Name', 'Payment Typology 3']]

# Generate counterfactuals
cf_reg = exp_reg.generate_counterfactuals(
    query_instance_dice,
    total_CFs=3,
    desired_range=[2000, 5000],
    features_to_vary=safe_features_to_vary
)

# Visualize and save
cf_reg.visualize_as_dataframe()
print(cf_reg.cf_examples_list[0].final_cfs_df)

with open("counterfactual_explanation.txt", "w") as f:
    f.write(cf_reg.cf_examples_list[0].final_cfs_df.to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000760 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 700
[LightGBM] [Info] Number of data points in the train set: 10545, number of used features: 30
[LightGBM] [Info] Start training from score 15124.345406


  0%|          | 0/1 [00:00<?, ?it/s]


ValueError: ('Feature', 'Hospital Service Area', 'has a value outside the dataset.')